In [6]:
# Setup: mount Drive, cd into the project, import shared utilities.
import sys
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount("/content/drive", force_remount=True)
%cd /content/drive/MyDrive/volatility-forecast

for m in list(sys.modules):   # Colab caches imports; drop stale src modules
    if m.startswith("src"):
        del sys.modules[m]

from src.data import load_work_frame
from src.splits import walk_forward_splits
from src.metrics import rmse

work = load_work_frame("data/processed/dataset.csv")
print("work shape:", work.shape)


Mounted at /content/drive
/content/drive/MyDrive/volatility-forecast
work shape: (3831, 21)


In [7]:
# --- windowing: univariate rv1 -> (samples, L, 1) + multi-horizon targets ---
L = 21                                    # lookback (matches HAR monthly span)
HORIZONS = ["y_rv1", "y_rv5", "y_rv21"]   # shared-backbone multi-output targets

def make_windows(work, feat_col="rv1", target_cols=HORIZONS, L=L):
    """
    Sliding-window builder for a univariate feature series.
    Sample ending at row i uses feat[i-L+1 : i+1] (L past values, incl. i) and
    the forward targets already aligned at row i (built in notebook 02).
    No future feature enters a window -> windowing is leak-free.
    Normalization is fold-local and applied later, NOT here.

    Returns
        X       : (n_samples, L, 1) float32   raw (unscaled) windows
        Y       : (n_samples, n_targets) float32
        end_idx : (n_samples,) int            positional row index of each window's last step
    """
    feat = work[feat_col].to_numpy(dtype=np.float64)
    Y_full = work[target_cols].to_numpy(dtype=np.float64)
    n = feat.shape[0]

    end_positions = np.arange(L - 1, n)                              # first full window ends at L-1
    X = np.stack([feat[i - L + 1 : i + 1] for i in end_positions])  # (n_samples, L)
    X = X[:, :, None].astype(np.float32)                            # -> (n_samples, L, 1)
    Y = Y_full[end_positions].astype(np.float32)                    # (n_samples, n_targets)
    return X, Y, end_positions

In [8]:
# --- sanity check: shapes + alignment ---
X, Y, end_idx = make_windows(work)
print("X shape  :", X.shape)             # expect (3811, 21, 1)
print("Y shape  :", Y.shape)             # expect (3811, 3)
print("end_idx  :", end_idx.min(), "..", end_idx.max())  # 20 .. 3830

i = 100                                   # arbitrary sample
assert np.isclose(X[i, -1, 0], work["rv1"].iloc[end_idx[i]])     # window tail == row's rv1
assert np.isclose(Y[i, 0],     work["y_rv1"].iloc[end_idx[i]])   # target == row's y_rv1
print("alignment OK")

X shape  : (3811, 21, 1)
Y shape  : (3811, 3)
end_idx  : 20 .. 3830
alignment OK


In [9]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
print("device:", DEVICE)

def set_seed(s=SEED):
    np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

class LSTMForecaster(nn.Module):
    """Univariate LSTM, shared backbone -> multi-horizon head (h1/h5/h21)."""
    def __init__(self, input_size=1, hidden_size=32, num_layers=1, n_targets=3, dropout=0.0):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True, dropout=dropout)
        self.head = nn.Linear(hidden_size, n_targets)   # shared backbone -> 3 horizons
    def forward(self, x):                # x: (B, L, 1)
        out, _ = self.lstm(x)
        last = out[:, -1, :]             # (B, hidden) last-step hidden state
        return self.head(last)           # (B, 3)

def scale_windows(Xw, scaler):
    n, L, f = Xw.shape
    return scaler.transform(Xw.reshape(-1, f)).reshape(n, L, f).astype(np.float32)

device: cpu


In [10]:
# --- per-fold prep: map fold rows -> window samples, temporal val split, leak-free scaling ---
def prep_fold(train_rows, test_rows, val_frac=0.15):
    tr_mask = np.isin(end_idx, train_rows)   # window belongs to train iff its end-row is a train row
    te_mask = np.isin(end_idx, test_rows)
    Xtr_raw, Ytr_raw = X[tr_mask], Y[tr_mask]   # np.isin preserves temporal order of end_idx
    Xte_raw, Yte_raw = X[te_mask], Y[te_mask]

    n_val = int(len(Xtr_raw) * val_frac)        # last (most recent) chunk of train -> val
    Xtr_raw, Xval_raw = Xtr_raw[:-n_val], Xtr_raw[-n_val:]
    Ytr_raw, Yval_raw = Ytr_raw[:-n_val], Ytr_raw[-n_val:]

    x_scaler = StandardScaler().fit(Xtr_raw.reshape(-1, 1))   # FIT ON TRAIN ONLY
    y_scaler = StandardScaler().fit(Ytr_raw)                  # per-horizon (per-column)

    Xtr = scale_windows(Xtr_raw, x_scaler); Ytr = y_scaler.transform(Ytr_raw).astype(np.float32)
    Xval = scale_windows(Xval_raw, x_scaler); Yval = y_scaler.transform(Yval_raw).astype(np.float32)
    Xte = scale_windows(Xte_raw, x_scaler)   # Yte stays RAW for honest RMSE
    return Xtr, Ytr, Xval, Yval, Xte, Yte_raw, y_scaler

def make_loader(Xa, Ya, batch=64, shuffle=False):
    ds = TensorDataset(torch.from_numpy(Xa), torch.from_numpy(Ya))
    return DataLoader(ds, batch_size=batch, shuffle=shuffle)

def train_fold(Xtr, Ytr, Xval, Yval, hidden=32, lr=1e-3, weight_decay=1e-4,
               max_epochs=200, patience=15, batch=64):
    set_seed()
    model = LSTMForecaster(hidden_size=hidden).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)  # weight_decay = L2 reg
    loss_fn = nn.MSELoss()
    tr_loader = make_loader(Xtr, Ytr, batch, shuffle=True)   # shuffle OK: samples are self-contained
    Xval_t, Yval_t = torch.from_numpy(Xval).to(DEVICE), torch.from_numpy(Yval).to(DEVICE)

    best_val, best_state, wait, hist = float("inf"), None, 0, []
    for epoch in range(max_epochs):
        model.train()
        for xb, yb in tr_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss_fn(model(xb), yb).backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(Xval_t), Yval_t).item()
        hist.append(val_loss)
        if val_loss < best_val - 1e-6:                       # early stopping: track best
            best_val, wait = val_loss, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience:
                break
    model.load_state_dict(best_state)                        # restore best weights, not last
    return model, hist, epoch + 1

def eval_fold(model, Xte, Yte_raw, y_scaler):
    model.eval()
    with torch.no_grad():
        pred_scaled = model(torch.from_numpy(Xte).to(DEVICE)).cpu().numpy()
    pred = y_scaler.inverse_transform(pred_scaled)           # back to RV units
    return {h: rmse(Yte_raw[:, j], pred[:, j]) for j, h in enumerate(HORIZONS)}

In [11]:
# --- smoke test: fold 0 only (verify machine + sane numbers before 11-fold loop) ---
folds = list(walk_forward_splits(len(work)))
print("n folds:", len(folds))            # expect 11

train_rows, test_rows = folds[0]
Xtr, Ytr, Xval, Yval, Xte, Yte_raw, y_scaler = prep_fold(train_rows, test_rows)
print("train/val/test windows:", len(Xtr), len(Xval), len(Xte))

model, hist, stopped = train_fold(Xtr, Ytr, Xval, Yval)
print(f"stopped @ epoch {stopped}, best val(scaled) {min(hist):.4f}")
print("fold0 test RMSE (RV units):", eval_fold(model, Xte, Yte_raw, y_scaler))

n folds: 11
train/val/test windows: 822 145 252
stopped @ epoch 28, best val(scaled) 0.5039
fold0 test RMSE (RV units): {'y_rv1': 0.008271342122827124, 'y_rv5': 0.0054291759875662255, 'y_rv21': 0.004459861032196605}


In [12]:
# Full 11-fold walk-forward for the LSTM.
def run_walk_forward(folds):
    rows = []
    for k, (train_rows, test_rows) in enumerate(folds):
        Xtr, Ytr, Xval, Yval, Xte, Yte_raw, y_scaler = prep_fold(train_rows, test_rows)
        model, hist, stopped = train_fold(Xtr, Ytr, Xval, Yval)
        fold_rmse = eval_fold(model, Xte, Yte_raw, y_scaler)
        fold_rmse.update({"fold": k, "stopped_epoch": stopped, "n_test": len(Xte)})
        rows.append(fold_rmse)
        print(f"f{k:02d} | stop@{stopped:3d} | "
              f"h1 {fold_rmse['y_rv1']:.4f}  h5 {fold_rmse['y_rv5']:.4f}  h21 {fold_rmse['y_rv21']:.4f}")
    df = pd.DataFrame(rows).set_index("fold")
    return df[["y_rv1", "y_rv5", "y_rv21", "stopped_epoch", "n_test"]]

lstm_results = run_walk_forward(folds)
print("\n--- LSTM per-fold RMSE (RV units) ---")
print(lstm_results.round(4))
print("\nfold-mean RMSE:", lstm_results[["y_rv1", "y_rv5", "y_rv21"]].mean().round(4).to_dict())

f00 | stop@ 28 | h1 0.0083  h5 0.0054  h21 0.0045
f01 | stop@ 18 | h1 0.0059  h5 0.0041  h21 0.0033
f02 | stop@ 19 | h1 0.0055  h5 0.0042  h21 0.0034
f03 | stop@ 18 | h1 0.0097  h5 0.0062  h21 0.0053
f04 | stop@ 40 | h1 0.0062  h5 0.0040  h21 0.0043
f05 | stop@ 26 | h1 0.0161  h5 0.0128  h21 0.0130
f06 | stop@ 46 | h1 0.0076  h5 0.0045  h21 0.0042
f07 | stop@ 36 | h1 0.0124  h5 0.0066  h21 0.0054
f08 | stop@ 17 | h1 0.0067  h5 0.0034  h21 0.0019
f09 | stop@ 17 | h1 0.0076  h5 0.0040  h21 0.0031
f10 | stop@ 16 | h1 0.0108  h5 0.0084  h21 0.0068

--- LSTM per-fold RMSE (RV units) ---
       y_rv1   y_rv5  y_rv21  stopped_epoch  n_test
fold                                               
0     0.0083  0.0054  0.0045             28     252
1     0.0059  0.0041  0.0033             18     252
2     0.0055  0.0042  0.0034             19     252
3     0.0097  0.0062  0.0053             18     252
4     0.0062  0.0040  0.0043             40     252
5     0.0161  0.0128  0.0130             26    

In [13]:
# --- annotate the two diagnostic folds vs the 03 baselines (eyeball check) ---
# 03 fold-mean baselines (from notebook 03):
#   persistence: h1 0.0117 / h5 0.0064 / h21 0.0058
#   HAR        : h1 0.0087 / h5 0.0055 / h21 0.0049
print("f05 (COVID, virtual drift) :", lstm_results.loc[5, ["y_rv1","y_rv5","y_rv21"]].round(4).to_dict())
print("f07 (2022 bear, REAL drift):", lstm_results.loc[7, ["y_rv1","y_rv5","y_rv21"]].round(4).to_dict())

f05 (COVID, virtual drift) : {'y_rv1': 0.0161, 'y_rv5': 0.0128, 'y_rv21': 0.013}
f07 (2022 bear, REAL drift): {'y_rv1': 0.0124, 'y_rv5': 0.0066, 'y_rv21': 0.0054}


In [15]:
# Per-fold h21 RMSE from notebook 03
persist_h21 = pd.Series({
    0: 0.0050, 1: 0.0041, 2: 0.0036, 3: 0.0061, 4: 0.0050,
    5: 0.0146, 6: 0.0043, 7: 0.0045, 8: 0.0022, 9: 0.0038, 10: 0.0084
})

har_h21 = pd.Series({
    0: 0.0043, 1: 0.0033, 2: 0.0034, 3: 0.0052, 4: 0.0040,
    5: 0.0122, 6: 0.0040, 7: 0.0056, 8: 0.0020, 9: 0.0031, 10: 0.0068
})

garch_h21 = pd.Series({
    0: 0.0042, 1: 0.0035, 2: 0.0037, 3: 0.0051, 4: 0.0039,
    5: 0.0117, 6: 0.0039, 7: 0.0051, 8: 0.0021, 9: 0.0032, 10: 0.0072
})

lstm_h21 = lstm_results["y_rv21"]

cmp = pd.DataFrame({
    "persist": persist_h21,
    "HAR": har_h21,
    "GARCH": garch_h21,
    "LSTM": lstm_h21,
})

cmp["winner"] = cmp[["persist", "HAR", "GARCH", "LSTM"]].idxmin(axis=1)
for m in ["HAR", "GARCH", "LSTM"]:
    cmp[f"{m}_beats_naive"] = cmp[m] < cmp["persist"]

print("\n=== 3-way per-fold h21 comparison (RMSE, RV units) ===")
print(cmp[["persist", "HAR", "GARCH", "LSTM", "winner"]].round(4))
print("\n--- Beats persistence baseline? ---")
print(cmp[["HAR_beats_naive", "GARCH_beats_naive", "LSTM_beats_naive"]])
print("\nfold-mean RMSE:")
print(cmp[["persist", "HAR", "GARCH", "LSTM"]].mean().round(4).to_dict())
print("\nf05 (COVID, vol↑ but P(Y|X) stable):")
print(cmp.loc[5, ["persist", "HAR", "GARCH", "LSTM", "winner"]])
print("\nf07 (2022 bear, REAL concept drift P(Y|X) shift):")
print(cmp.loc[7, ["persist", "HAR", "GARCH", "LSTM", "winner"]])
print(f"\n→ f07: parametric models lose to naive? "
      f"HAR={not cmp.loc[7, 'HAR_beats_naive']}, "
      f"GARCH={not cmp.loc[7, 'GARCH_beats_naive']}, "
      f"LSTM={not cmp.loc[7, 'LSTM_beats_naive']}")


=== 3-way per-fold h21 comparison (RMSE, RV units) ===
    persist     HAR   GARCH    LSTM   winner
0    0.0050  0.0043  0.0042  0.0045    GARCH
1    0.0041  0.0033  0.0035  0.0033      HAR
2    0.0036  0.0034  0.0037  0.0034      HAR
3    0.0061  0.0052  0.0051  0.0053    GARCH
4    0.0050  0.0040  0.0039  0.0043    GARCH
5    0.0146  0.0122  0.0117  0.0130    GARCH
6    0.0043  0.0040  0.0039  0.0042    GARCH
7    0.0045  0.0056  0.0051  0.0054  persist
8    0.0022  0.0020  0.0021  0.0019     LSTM
9    0.0038  0.0031  0.0032  0.0031      HAR
10   0.0084  0.0068  0.0072  0.0068      HAR

--- Beats persistence baseline? ---
    HAR_beats_naive  GARCH_beats_naive  LSTM_beats_naive
0              True               True              True
1              True               True              True
2              True              False              True
3              True               True              True
4              True               True              True
5              True      